# AmbiFix legacy-corpus grouped exploratory BERT — RTX 5060 Ti 16 GB

**New benchmark only. No training has been executed in this notebook.**

This single fixed BERT experiment consumes only the **new package's** `data/train.csv` and `data/validation.csv`. It never loads the repository's frozen clean splits, raw corpus, or historical `final.csv`. A quality gate and input hashes must pass before GPU initialization or model downloads.

**Critical scientific boundary:** the source corpus can contain requirements from the original frozen test. A model trained here is **ineligible for evaluation on that original frozen test**. Never use this model as a clean candidate for the original benchmark. Results are not directly comparable to the old validation macro F1 of 0.581140.

Candidate families are conservative **co-isolation constraints**, not verified back-translation parentage. A partial recovery can exclude all unlinked variants; that subset **cannot test whether the full historical augmentation improves BERT**. Residual semantic-family recall is unknown. Neither grouping nor a passing operational gate establishes a fully independent or leakage-free benchmark.

## GPU collaborator instructions

1. Extract the entire approved GPU ZIP into a new directory; preserve the included manifest and protocol.
2. Open this notebook from the package root or its `notebooks/` directory. Run cells in order once. Do not add old split files or historical fine-tuned checkpoints.
3. Use the optional setup commands only on the GPU machine if its environment needs them; restart the kernel afterward. Never run training on the author's audit machine.
4. Return the executed notebook, the complete new run output directory and the **best_model/** folder. Do not send or inspect any original held-out test.
5. Keep the fixed training settings below. If a gate, CUDA check, data check or runtime fails, return the error and partial logs; do not change data/grouping/thresholds to improve validation scores.

The notebook starts from the official original pretrained `bert-base-uncased` repository, resolves an immutable revision at run time, and never warm-starts from a historical fine-tuned model. One fixed square-root-weighted loss is used; no comparison/search loop is present.

## Public source release: local preparation is required

No research records, generated quality reports, model weights, or ready-made package manifest are distributed here. Run All deliberately fails closed until authorized local development inputs are restored and verified. Reconstruct the separate partial-recovery benchmark using `scripts/build_benchmark.py` and permitted local source data; its actual passing grouping report must exist. This is not the original frozen benchmark and is never eligible for its final Test.

From this experiment directory, run the source-only metadata helper after preparing those inputs:

```text
python scripts/prepare_local_manifest.py --root . --profile legacy --authorized-local-development-data --execute
```

This helper records input hashes only; it never copies records, packages a dataset, trains, downloads a model, or authorizes Test. It does not replace the notebook's unchanged data/label/weight/lineage checks. See `LOCAL_REPRODUCTION.md`.


## 0. Optional GPU-only environment setup

The following is a **suggested pinned environment**, not a GPU runtime verified during notebook preparation. The CUDA architecture/version and finite-matrix preflight below must pass before downloading original model weights. These commands are **comments and are not executed automatically**. Use only if needed on the collaborator's Windows/Linux GPU machine, then restart the kernel. Exact runtime versions will be exported with the run.

```text
python -m pip install torch==2.9.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install transformers==4.51.3 pandas==2.2.3 numpy==1.26.4 scikit-learn==1.6.1 safetensors==0.5.3 huggingface_hub
```

Start the kernel with `PYTHONHASHSEED=42` if feasible. This cannot be retroactively guaranteed merely by setting the variable inside a running Python process. All algorithmic seeds/data ordering below are explicit.

In [ ]:
# Only stdlib plus the small tabular stack is loaded before the package checks.
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
import hashlib
import json
import os
import platform
import random
import re
import subprocess
import sys
import unicodedata
import numpy as np
import pandas as pd

REQUIRED_DATA_COLUMNS = ["row_id", "group_id", "Text", "Class"]
REQUIRED_HASHED_INPUTS = ["data/train.csv", "data/validation.csv", "config/protocol.json", "reports/QUALITY_GATE.json"]
PERMITTED_INPUTS = set(REQUIRED_HASHED_INPUTS + ["package_manifest.json"])
ALLOWED_GATE_STATUSES = {"PASS_PARTIAL_RECOVERY", "PASS_GROUPED_EXPLORATORY"}
DATASET_CLASS_IDS = [1, 2, 3, 4, 5, 6]


def check_quality_gate(gate):
    if gate.get("status") not in ALLOWED_GATE_STATUSES:
        raise ValueError("Grouping quality gate does not permit this GPU experiment.")
    if gate.get("approved_for_gpu_package") is not True:
        raise ValueError("GPU packaging/training was not approved by the experiment quality gate.")
    if gate.get("eligible_for_original_frozen_test") is not False:
        raise ValueError("This experiment must explicitly be ineligible for the original frozen test.")
    return True


def check_protocol_binding(protocol, gate, verified_hashes):
    benchmark_id = protocol.get("benchmark_id")
    if not isinstance(benchmark_id, str) or not benchmark_id.strip():
        raise ValueError("Protocol must identify this new benchmark explicitly")
    if gate.get("benchmark_id") != benchmark_id:
        raise ValueError("Quality gate belongs to a different benchmark")
    if protocol.get("eligible_for_original_frozen_test") is not False:
        raise ValueError("Protocol must explicitly prohibit original frozen-test evaluation")
    protocol_digest = verified_hashes.get("config/protocol.json")
    if not isinstance(protocol_digest, str) or not re.fullmatch(r"[0-9a-f]{64}", protocol_digest):
        raise ValueError("Verified protocol SHA-256 is missing or invalid")
    if gate.get("protocol_sha256") != protocol_digest:
        raise ValueError("Quality gate is not bound to the verified protocol bytes")
    return benchmark_id


def check_cuda_build(capability, cuda_version, architecture_list):
    capability = tuple(int(x) for x in capability)
    if len(capability) != 2:
        raise ValueError("Unexpected GPU compute capability")
    if capability[0] >= 12:
        try:
            version = tuple(int(x) for x in str(cuda_version).split(".")[:2])
        except (TypeError, ValueError):
            raise ValueError("Cannot identify CUDA build version for this Blackwell GPU")
        if len(version) != 2 or version < (12, 8):
            raise ValueError("Blackwell requires this recipe's CUDA 12.8+ PyTorch build")
        required_arch = f"sm_{capability[0]}{capability[1]}"
        if required_arch not in architecture_list:
            raise ValueError(f"Installed PyTorch wheel does not list required architecture {required_arch}")
    return {"compute_capability": list(capability), "cuda_build": cuda_version,
            "compiled_architectures": list(architecture_list), "metadata_preflight_passed": True}


def package_file(package_root, relative):
    if relative not in PERMITTED_INPUTS:
        raise ValueError(f"This notebook may not read that input: {relative}")
    parts = PurePosixPath(relative).parts
    if any(part.lower() == "test.csv" for part in parts) or ".." in parts:
        raise ValueError("Forbidden input path")
    resolved = (package_root / relative).resolve()
    if not resolved.is_relative_to(package_root.resolve()):
        raise ValueError("Input escapes package root")
    return resolved


def manifest_hash_map(manifest):
    records = manifest.get("files")
    if not isinstance(records, list):
        raise ValueError("package_manifest.json must contain a files list of path/sha256 records")
    mapping = {}
    for record in records:
        relative, digest = record.get("path"), record.get("sha256")
        if not isinstance(relative, str) or not isinstance(digest, str):
            raise ValueError("Malformed package manifest entry")
        parts = PurePosixPath(relative).parts
        if PurePosixPath(relative).is_absolute() or ".." in parts or "\\" in relative:
            raise ValueError("Unsafe path in package manifest")
        if any(part.lower() == "test.csv" for part in parts):
            raise ValueError("A test file must not be included in this package")
        if relative in mapping or not re.fullmatch(r"[0-9a-fA-F]{64}", digest):
            raise ValueError("Duplicate path or invalid SHA-256 in package manifest")
        mapping[relative] = digest.lower()
    if not set(REQUIRED_HASHED_INPUTS).issubset(mapping):
        raise ValueError("Manifest does not bind all four required data/protocol/gate inputs")
    return mapping


def normalize_for_integrity(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(text))).strip().casefold()


def validate_new_partitions(train_frame, validation_frame):
    for name, frame in [("train", train_frame), ("validation", validation_frame)]:
        if not set(REQUIRED_DATA_COLUMNS).issubset(frame.columns):
            raise ValueError(f"{name}: expected {REQUIRED_DATA_COLUMNS}")
        if frame[REQUIRED_DATA_COLUMNS].isna().any().any():
            raise ValueError(f"{name}: missing required value")
        if frame.empty or not frame.row_id.is_unique:
            raise ValueError(f"{name}: empty partition or repeated row_id")
        if frame.row_id.astype(str).str.strip().eq("").any() or frame.group_id.astype(str).str.strip().eq("").any():
            raise ValueError(f"{name}: blank row/group ID")
        if not np.array_equal(frame.Class.to_numpy(), frame.Class.astype(int).to_numpy()):
            raise ValueError(f"{name}: noninteger class IDs")
        if set(frame.Class.astype(int)) != set(DATASET_CLASS_IDS):
            raise ValueError(f"{name}: all six dataset classes 1..6 are required")
        if frame.Text.map(normalize_for_integrity).eq("").any():
            raise ValueError(f"{name}: blank normalized text")
        if frame.groupby("group_id").Class.nunique().gt(1).any():
            raise ValueError(f"{name}: cross-label candidate family escaped quarantine")
    row_overlap = set(train_frame.row_id) & set(validation_frame.row_id)
    group_overlap = set(train_frame.group_id) & set(validation_frame.group_id)
    normalized_overlap = set(train_frame.Text.map(normalize_for_integrity)) & set(validation_frame.Text.map(normalize_for_integrity))
    if row_overlap or group_overlap or normalized_overlap:
        raise ValueError("New partitions overlap by row_id, group_id or normalized Text")
    return {"row_id_overlap": 0, "group_id_overlap": 0, "normalized_text_overlap": 0,
            "train_rows": len(train_frame), "validation_rows": len(validation_frame),
            "train_groups": int(train_frame.group_id.nunique()),
            "validation_groups": int(validation_frame.group_id.nunique()),
            "train_class_counts": {str(k): int(v) for k, v in train_frame.Class.value_counts().sort_index().items()},
            "validation_class_counts": {str(k): int(v) for k, v in validation_frame.Class.value_counts().sort_index().items()},
            "note": "Operational integrity checks do not establish verified original-parent lineage or semantic independence."}

## 1. Locate and verify only the approved package

No corpus or frozen original split is searched. The manifest binds both new data partitions, the fixed grouping protocol and the quality gate. A passing gate is permission for this qualified exploratory/partial-recovery benchmark, not proof of complete family provenance.

In [ ]:
# Run from the extracted package root or its notebooks/ subdirectory only.
_cwd = Path.cwd().resolve()
_candidates = [_cwd]
if _cwd.name == "notebooks":
    _candidates.append(_cwd.parent)
_candidates = [p for p in _candidates if (p / "package_manifest.json").is_file() and (p / "config/protocol.json").is_file()]
if len(_candidates) != 1:
    raise FileNotFoundError("Open this notebook within the extracted approved package root or its notebooks/ directory.")
PACKAGE_ROOT = _candidates[0]
manifest = json.loads(package_file(PACKAGE_ROOT, "package_manifest.json").read_text(encoding="utf-8-sig"))
manifest_hashes = manifest_hash_map(manifest)
verified_input_hashes = {}
for relative in REQUIRED_HASHED_INPUTS:
    digest = hashlib.sha256(package_file(PACKAGE_ROOT, relative).read_bytes()).hexdigest()
    if digest != manifest_hashes[relative]:
        raise ValueError(f"Package input hash mismatch: {relative}")
    verified_input_hashes[relative] = digest
protocol = json.loads(package_file(PACKAGE_ROOT, "config/protocol.json").read_text(encoding="utf-8-sig"))
quality_gate = json.loads(package_file(PACKAGE_ROOT, "reports/QUALITY_GATE.json").read_text(encoding="utf-8-sig"))
check_quality_gate(quality_gate)
BENCHMARK_ID = check_protocol_binding(protocol, quality_gate, verified_input_hashes)
train_df = pd.read_csv(package_file(PACKAGE_ROOT, "data/train.csv"), dtype={"row_id": str, "group_id": str})
val_df = pd.read_csv(package_file(PACKAGE_ROOT, "data/validation.csv"), dtype={"row_id": str, "group_id": str})
integrity = validate_new_partitions(train_df, val_df)
print("NEW BENCHMARK:", BENCHMARK_ID)
print(json.dumps(integrity, indent=2))
print("Quality gate:", quality_gate["status"])
print("Full 25k provenance pass:", quality_gate.get("full_25k_provenance_pass", "not established"))
print("Original frozen-test eligibility: FALSE. This model must never be evaluated on that test.")
print("If partial recovery excludes unlinked variants, this experiment does NOT establish the full historical augmentation effect.")

## 2. Fixed training recipe and immutable original pretrained revision

One fixed configuration is used. Training class weights are calculated exclusively from the new training partition. Minimum-improvement early stopping is explicit: a checkpoint is replaced only when validation macro F1 improves by more than 0.001; it need not be the literal maximum if a later smaller gain occurs. Both selected and absolute-maximum logged epochs are recorded.

The pretrained revision is resolved from the official BERT repository before initialization and then used immutably for model/tokenizer loading. Network access and original pretrained downloads occur only when the collaborator runs these GPU cells. The local preparation/static validation did neither.

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    MODEL_ID: str = "bert-base-uncased"
    SEED: int = 42
    MAX_LENGTH: int = 192
    TRAIN_BATCH_SIZE: int = 8
    EVAL_BATCH_SIZE: int = 16
    GRADIENT_ACCUMULATION: int = 2
    MAX_EPOCHS: int = 20
    MIN_EPOCHS: int = 4
    EARLY_STOP_PATIENCE: int = 4
    MIN_DELTA: float = 0.001
    LR: float = 2e-5
    WEIGHT_DECAY: float = 0.01
    LR_PLATEAU_PATIENCE: int = 2
    LR_PLATEAU_FACTOR: float = 0.5
    MIN_LR: float = 2e-7
    MAX_GRAD_NORM: float = 1.0
    CLASS_WEIGHT_POWER: float = 0.5

cfg = TrainingConfig()
# Must be set before importing/initializing CUDA. Interpreter PYTHONHASHSEED is recorded separately.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
import transformers
import sklearn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import HfApi
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this GPU notebook; do not train on the audit machine.")
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_gib < 14:
    raise RuntimeError("This fixed recipe expects approximately 16 GB GPU memory; return an error rather than silently changing the experiment.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 is required by this fixed RTX 5060 Ti recipe; report unsupported hardware rather than silently changing precision.")
AMP_DTYPE = torch.bfloat16
cuda_preflight = check_cuda_build(torch.cuda.get_device_capability(0), torch.version.cuda, torch.cuda.get_arch_list())
# Tiny BF16 CUDA kernel check BEFORE Hub metadata resolution or pretrained downloads.
with torch.no_grad():
    _cuda_probe = torch.randn(64, 64, device=device, dtype=AMP_DTYPE)
    _cuda_product = _cuda_probe @ _cuda_probe.T
    if not torch.isfinite(_cuda_product).all().item():
        raise FloatingPointError("CUDA finite-matrix smoke check failed")
    torch.cuda.synchronize()
    del _cuda_probe, _cuda_product
torch.cuda.empty_cache()
cuda_preflight["finite_bfloat16_matmul_passed"] = True
print("CUDA preflight:", json.dumps(cuda_preflight))


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_all(cfg.SEED)  # BEFORE tokenizer, dataloader or model initialization.
if cfg.MODEL_ID != "bert-base-uncased" or Path(cfg.MODEL_ID).exists() or (PACKAGE_ROOT / cfg.MODEL_ID).exists():
    raise ValueError("Only the official original pretrained BERT repository is allowed; no local or historically fine-tuned initialization.")
pretrained_info = HfApi().model_info(cfg.MODEL_ID, revision="main")
if pretrained_info.id not in {"bert-base-uncased", "google-bert/bert-base-uncased"}:
    raise ValueError("Unexpected pretrained repository identity")
PRETRAINED_REVISION = pretrained_info.sha
if not re.fullmatch(r"[0-9a-f]{40}", PRETRAINED_REVISION or ""):
    raise ValueError("Could not resolve an immutable original pretrained revision")
print("Original pretrained repository:", pretrained_info.id, "revision:", PRETRAINED_REVISION)
print("GPU:", gpu_name, "GiB:", round(gpu_gib, 2), "BF16:", True)

# Semantic names are optional metadata; dataset/model numeric mapping is fixed.
_supplied_names = protocol.get("class_names", {})
CLASS_NAMES = {i: str(_supplied_names.get(str(i), _supplied_names.get(i, f"dataset_class_{i}"))) for i in DATASET_CLASS_IDS}
ID2LABEL = {i-1: CLASS_NAMES[i] for i in DATASET_CLASS_IDS}
LABEL2ID = {name: i for i, name in ID2LABEL.items()}
if len(LABEL2ID) != 6:
    raise ValueError("Display label names must remain distinct; do not merge dataset classes")

SESSION = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
RUN_DIR = PACKAGE_ROOT / "outputs" / f"legacy_grouped_bert_sqrtweighted_{SESSION}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
BEST_DIR = RUN_DIR / "best_model"
(RUN_DIR / "training_config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
(RUN_DIR / "group_overlap_check.json").write_text(json.dumps(integrity, indent=2), encoding="utf-8")
(RUN_DIR / "input_protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
(RUN_DIR / "input_quality_gate.json").write_text(json.dumps(quality_gate, indent=2), encoding="utf-8")
print("NEW output folder:", RUN_DIR)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_ID, revision=PRETRAINED_REVISION, use_fast=True)


class RequirementsDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame.Text.astype(str).tolist()
        self.labels = (frame.Class.astype(int) - 1).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.texts[index], self.labels[index]


def collate(items):
    texts, labels = zip(*items)
    features = tokenizer(list(texts), padding=True, truncation=True,
                         max_length=cfg.MAX_LENGTH, return_tensors="pt")
    features["labels"] = torch.tensor(labels, dtype=torch.long)
    return features


train_ds = RequirementsDataset(train_df)
val_ds = RequirementsDataset(val_df)
train_generator = torch.Generator().manual_seed(cfg.SEED)
train_loader = DataLoader(train_ds, batch_size=cfg.TRAIN_BATCH_SIZE, shuffle=True,
                          generator=train_generator, num_workers=0, pin_memory=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=cfg.EVAL_BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True, collate_fn=collate)
train_counts = np.bincount(train_ds.labels, minlength=6).astype(np.float64)
class_weights = (train_counts.sum() / (6.0 * train_counts)) ** cfg.CLASS_WEIGHT_POWER
class_weights /= class_weights.mean()
weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
(RUN_DIR / "class_weights.json").write_text(json.dumps({
    "source_partition": "NEW training partition only", "dataset_class_order": DATASET_CLASS_IDS,
    "model_index_order": list(range(6)), "training_counts": train_counts.astype(int).tolist(),
    "formula": "(N_train / (6 * class_count))**0.5, normalized to arithmetic mean 1",
    "values": class_weights.tolist(),
    "loss_reduction": "weighted sum normalized by sum of true-class weights across each gradient-accumulation window"
}, indent=2), encoding="utf-8")

# A new six-class head is initialized from ORIGINAL pretrained BERT, never an old fine-tuned artifact.
seed_all(cfg.SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.MODEL_ID, revision=PRETRAINED_REVISION, num_labels=6,
    id2label=ID2LABEL, label2id=LABEL2ID, attn_implementation="eager"
).to(device)
if model.config.model_type != "bert":
    raise ValueError("Unexpected model architecture")
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=cfg.LR_PLATEAU_FACTOR,
    patience=cfg.LR_PLATEAU_PATIENCE, threshold=cfg.MIN_DELTA,
    threshold_mode="abs", min_lr=cfg.MIN_LR
)
# BF16 does not require loss scaling; the same API keeps optimizer accounting explicit.
scaler = torch.amp.GradScaler("cuda", enabled=False)
model.config.to_json_file(RUN_DIR / "initial_model_config.json")
print("Training-only counts:", train_counts.astype(int).tolist())
print("Training-only class weights:", class_weights.tolist())

## 3. Validation utilities

Evaluation always uses `model.eval()` and inference mode. Validation loss is unweighted cross entropy; primary early stopping uses explicit six-class **macro F1**. Weighted F1 and accuracy are separately reported. No labels/thresholds/grouping rules are revised from validation scores.

In [ ]:
@torch.inference_mode()
def evaluate(current_model, loader, return_probabilities=False):
    current_model.eval()
    labels_all, predictions_all, probability_parts = [], [], []
    loss_sum, count = 0.0, 0
    for batch in loader:
        target = batch["labels"].to(device)
        features = {k: t.to(device, non_blocking=True) for k, t in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            logits = current_model(**features).logits
        logits32 = logits.float()
        if not torch.isfinite(logits32).all().item():
            raise FloatingPointError("Non-finite validation logits")
        batch_loss = torch.nn.functional.cross_entropy(logits32, target, reduction="sum")
        if not torch.isfinite(batch_loss).item():
            raise FloatingPointError("Non-finite validation loss")
        loss_sum += batch_loss.item()
        count += len(target)
        probability = torch.softmax(logits32, dim=-1)
        prediction = probability.argmax(dim=-1)
        labels_all.extend(target.cpu().tolist())
        predictions_all.extend(prediction.cpu().tolist())
        if return_probabilities:
            probability_parts.append(probability.cpu().numpy())
    scores = {
        "unweighted_loss": loss_sum / count,
        "accuracy": float(accuracy_score(labels_all, predictions_all)),
        "macro_f1": float(f1_score(labels_all, predictions_all, labels=list(range(6)), average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels_all, predictions_all, labels=list(range(6)), average="weighted", zero_division=0)),
    }
    probabilities = np.concatenate(probability_parts) if return_probabilities else None
    return scores, np.asarray(labels_all, dtype=int), np.asarray(predictions_all, dtype=int), probabilities

## 4. One fixed training run — GPU execution begins here

No training has been performed during notebook preparation. This cell trains only when the collaborator explicitly executes it after the package checks. A unique output folder is used. Early stopping saves the selected model/tokenizer and never changes grouping or split rules.

In [ ]:
import time
history = []
best_macro_f1, best_epoch, stale = float("-inf"), None, 0
run_started = time.time()

for epoch in range(1, cfg.MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_numerator = epoch_denominator = window_denominator = 0.0
    successful_optimizer_steps = 0
    epoch_started = time.time()
    for batch_number, batch in enumerate(train_loader, start=1):
        target = batch["labels"].to(device)
        features = {k: t.to(device, non_blocking=True) for k, t in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            logits = model(**features).logits
        numerator = torch.nn.functional.cross_entropy(logits.float(), target, weight=weights, reduction="sum")
        denominator = float(weights[target].sum().item())
        if not torch.isfinite(numerator).item() or denominator <= 0:
            raise FloatingPointError("Non-finite loss or invalid training-weight denominator")
        scaler.scale(numerator).backward()
        epoch_numerator += float(numerator.item())
        epoch_denominator += denominator
        window_denominator += denominator
        if batch_number % cfg.GRADIENT_ACCUMULATION == 0 or batch_number == len(train_loader):
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.div_(window_denominator)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM, error_if_nonfinite=True)
            scaler.step(optimizer)
            scaler.update()
            successful_optimizer_steps += 1
            optimizer.zero_grad(set_to_none=True)
            window_denominator = 0.0

    metrics, _, _, _ = evaluate(model, val_loader)
    if not np.isfinite(metrics["macro_f1"]):
        raise FloatingPointError("Non-finite validation macro F1")
    lr_before = optimizer.param_groups[0]["lr"]
    scheduler.step(metrics["macro_f1"])  # after actual optimizer steps and validation
    row = {"epoch": epoch, "train_weighted_loss": epoch_numerator / epoch_denominator,
           **{"validation_" + key: value for key, value in metrics.items()},
           "learning_rate_before_scheduler": lr_before,
           "learning_rate_after_scheduler": optimizer.param_groups[0]["lr"],
           "successful_optimizer_steps": successful_optimizer_steps,
           "epoch_seconds": time.time() - epoch_started}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "epoch_metrics.csv", index=False)
    if metrics["macro_f1"] > best_macro_f1 + cfg.MIN_DELTA:
        best_macro_f1, best_epoch, stale = metrics["macro_f1"], epoch, 0
        BEST_DIR.mkdir(exist_ok=True)
        model.save_pretrained(BEST_DIR, safe_serialization=True)
        tokenizer.save_pretrained(BEST_DIR)
        (RUN_DIR / "best_epoch.json").write_text(json.dumps({
            "selected_epoch": best_epoch, "selected_validation_macro_f1": best_macro_f1,
            "selection_rule": "replace only if macro F1 > previous selected macro F1 + MIN_DELTA",
            "minimum_improvement": cfg.MIN_DELTA
        }, indent=2), encoding="utf-8")
        message = "saved selected checkpoint"
    else:
        stale += 1
        message = f"no meaningful improvement ({stale}/{cfg.EARLY_STOP_PATIENCE})"
    print(f"epoch={epoch:02d} macro_f1={metrics['macro_f1']:.6f} weighted_f1={metrics['weighted_f1']:.6f} accuracy={metrics['accuracy']:.6f}; {message}", flush=True)
    if epoch >= cfg.MIN_EPOCHS and stale >= cfg.EARLY_STOP_PATIENCE:
        print("Early stopping; selected epoch:", best_epoch)
        break

if best_epoch is None or not BEST_DIR.is_dir():
    raise RuntimeError("No selected checkpoint was saved")
del model, optimizer, scheduler, scaler
torch.cuda.empty_cache()
# Only this NEW run's selected checkpoint is reloaded. Historical fine-tuned artifacts are never loaded.
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR, attn_implementation="eager").to(device)
final_metrics, y0, p0, probabilities = evaluate(best_model, val_loader, return_probabilities=True)
if abs(final_metrics["macro_f1"] - best_macro_f1) > 1e-10:
    raise RuntimeError("Reloaded selected checkpoint does not reproduce its saved validation macro F1")

## 5. Export validation evidence, checkpoints and provenance

Return all small reports plus `best_model/`, tokenizer/configuration files and this **executed** notebook. All results are specific to the new benchmark. `eligible_for_original_frozen_test` is always false. A passing grouping gate and this notebook do not certify complete parent lineage, semantic independence, expert synthetic-label validity or final-test performance.

In [ ]:
true_labels, predicted_labels = y0 + 1, p0 + 1
if not np.isfinite(probabilities).all() or np.max(np.abs(probabilities.sum(axis=1) - 1)) > 1e-5:
    raise FloatingPointError("Invalid exported validation probabilities")
if not np.array_equal(true_labels, val_df.Class.to_numpy(dtype=int)):
    raise ValueError("Validation row/label alignment changed during inference")
if not np.array_equal(probabilities.argmax(axis=1) + 1, predicted_labels):
    raise ValueError("Probability class order does not match predicted dataset IDs")
predictions = val_df[["row_id", "group_id", "Text", "Class"]].copy()
predictions["predicted_class"] = predicted_labels
predictions["correct"] = true_labels == predicted_labels
predictions["confidence"] = probabilities.max(axis=1)
for i in DATASET_CLASS_IDS:
    predictions[f"prob_class_{i}"] = probabilities[:, i-1]
predictions.to_csv(RUN_DIR / "validation_predictions.csv", index=False, encoding="utf-8-sig")
predictions.loc[~predictions.correct].to_csv(RUN_DIR / "validation_errors.csv", index=False, encoding="utf-8-sig")
cm = confusion_matrix(true_labels, predicted_labels, labels=DATASET_CLASS_IDS)
pd.DataFrame(cm, index=[f"true_{i}" for i in DATASET_CLASS_IDS],
             columns=[f"pred_{i}" for i in DATASET_CLASS_IDS]).to_csv(RUN_DIR / "confusion_matrix.csv")
report_dict = classification_report(true_labels, predicted_labels, labels=DATASET_CLASS_IDS,
                                    output_dict=True, zero_division=0)
report_text = classification_report(true_labels, predicted_labels, labels=DATASET_CLASS_IDS,
                                    target_names=[CLASS_NAMES[i] for i in DATASET_CLASS_IDS], digits=6, zero_division=0)
(RUN_DIR / "classification_report.json").write_text(json.dumps(report_dict, indent=2), encoding="utf-8")
(RUN_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
metrics_export = {"benchmark_id": BENCHMARK_ID, "scope": "NEW grouped exploratory validation only",
                  "validation_rows": len(val_df), "selected_epoch": best_epoch,
                  **{"validation_" + key: value for key, value in final_metrics.items()},
                  "dataset_class_order": DATASET_CLASS_IDS, "zero_division": 0,
                  "eligible_for_original_frozen_test": False}
(RUN_DIR / "validation_metrics.json").write_text(json.dumps(metrics_export, indent=2), encoding="utf-8")
with (RUN_DIR / "pip_freeze.txt").open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)

# Hash only files this run has created, never the original corpus or any frozen split.
checkpoint_hashes = {}
for output_file in sorted(BEST_DIR.iterdir()):
    if output_file.is_file():
        checkpoint_hashes[str(output_file.relative_to(RUN_DIR))] = hashlib.sha256(output_file.read_bytes()).hexdigest()
absolute_maximum = max(history, key=lambda item: item["validation_macro_f1"])
provenance = {
    "benchmark_id": BENCHMARK_ID, "run_id": RUN_DIR.name, "status": "completed_validation_only",
    "eligible_for_original_frozen_test": False,
    "eligibility_reason": "This new corpus-derived benchmark can contain original frozen-test requirements; never evaluate this trained model on that test.",
    "comparison_boundary": "Not directly comparable to old validation macro F1 0.581140; different benchmark/population and grouping.",
    "grouping_interpretation": "Candidate family co-isolation is not verified original-parent lineage; semantic recall is unknown.",
    "partial_recovery_caveat": "If unlinked augmented rows are excluded, this experiment cannot establish a full historical back-translation benefit.",
    "configuration": asdict(cfg), "input_sha256": verified_input_hashes,
    "quality_gate": quality_gate, "protocol": protocol, "group_integrity": integrity,
    "initialization": {"repository": pretrained_info.id, "requested_id": cfg.MODEL_ID,
                       "resolved_immutable_revision": PRETRAINED_REVISION,
                       "historical_finetuned_initialization_used": False},
    "dataset_to_model_label_mapping": {str(i): i-1 for i in DATASET_CLASS_IDS},
    "class_names": CLASS_NAMES, "class_weight_counts_training_only": train_counts.astype(int).tolist(),
    "class_weight_values": class_weights.tolist(), "selected_epoch": best_epoch,
    "epochs_completed": len(history), "selected_validation_metrics": final_metrics,
    "absolute_maximum_logged_epoch": int(absolute_maximum["epoch"]),
    "absolute_maximum_logged_macro_f1": float(absolute_maximum["validation_macro_f1"]),
    "checkpoint_sha256": checkpoint_hashes, "wall_seconds": time.time() - run_started,
    "python": sys.version, "platform": platform.platform(), "pythonhashseed_environment": os.environ.get("PYTHONHASHSEED"),
    "torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "gpu": gpu_name, "gpu_gib": gpu_gib, "compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda_preflight": cuda_preflight,
    "precision": "bfloat16 autocast with float32 losses", "determinism": "seeded ordering and deterministic-algorithm warnings enabled; bitwise cross-hardware identity not guaranteed",
    "test_data_read": False, "original_frozen_splits_read": False,
    "further_hyperparameter_search_performed": False,
}
(RUN_DIR / "experiment_provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print(report_text)
print("NEW validation metrics:", json.dumps(metrics_export, indent=2))
print("Return executed notebook, all reports and best_model/ from:", RUN_DIR)
print("DO NOT evaluate this model against the original frozen test or label these scores as that benchmark's results.")

## Return package checklist

- Executed `.ipynb` with its original outputs and any warnings/errors.
- `training_config.json`, `class_weights.json`, `group_overlap_check.json`, input protocol/gate snapshots and original package manifest.
- `epoch_metrics.csv`, `best_epoch.json`, all validation metrics/predictions/error/report/confusion files.
- `experiment_provenance.json`, `pip_freeze.txt`, and the full `best_model/` weights/config/tokenizer directory.

Do not claim an experiment completed unless the real run produced those files. If training fails, send its exact error and whatever actual artifacts exist. Do not fabricate placeholders, silently alter the split or recipe, add test data, or reuse the historical contaminated checkpoint.